AOI properties to compare:

- [ ] Land cover breakdown (to see potential for confusion between mangroves & dense vegetation)
- [ ] Slope class distribution (to see potential for confusion from topographic shadow effects)
- [ ] Mangrove patch fragmentation (to see effect of patch sizes on accuracy)
- [ ] Climate type (Taylor chart showing Type 1, 2, 3? determines which months to best get imagery)

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
import rioxarray as rxr
import rasterstats as rstat
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import xarray as xr
from xhistogram.xarray import histogram
from shapely.geometry import Polygon, MultiPolygon
from shapely import make_valid

# from odc.stac import stac_load, configure_s3_access
import stackstac as stk
import pystac_client
import planetary_computer as pc
from dask.distributed import Client
import dask
import dask.array as da
import dask.dataframe as dd
from pystac.extensions.projection import ProjectionExtension as proj

import json
from pathlib import Path

In [2]:
def extract_aoi_geom(fp):
    with open(fp, 'r') as f:
        geojson_dict = json.load(f)

    geom_dict = geojson_dict['features'][0]['geometry']

    return geom_dict

In [3]:
def extract_lulc_aoi(geom, catalog, collection_id, year=2021):
    search = catalog.search(
        collections=[collection_id],
        intersects=geom,
        datetime=f'{year}-01-01/{year}-12-31'
    )

    items = search.item_collection()

    if len(items) == 0:
        raise ValueError('Search in catalog returned no results')

    epsg = proj.ext(items[0]).epsg
    nodata = items[0].assets["data"].extra_fields["raster:bands"][0]["nodata"]

    da_lulc = stk.stack(
        items,
        epsg=epsg,
        dtype=np.int64,
        fill_value=0,
        rescale=False
    )

    da_lulc = stk.mosaic(
        da_lulc,
        dim="time",
        axis=None,
        nodata=nodata
    ).squeeze()

    return da_lulc

In [4]:
def chunk_zonal_stats(raster_chunk, zones_geoms, stats, transform):
    """Worker function applied to each Dask chunk"""
    result = rstat.zonal_stats(
        raster_chunk, zones_geoms, categorical=True, affine=transform
    )

    return np.array([])

In [5]:
LAND_COVER_CATEGORIES = {
    1: "Water",
    2: "Trees",
    3: "Flooded Vegetation",
    4: "Crops",
    5: "Built Area",
    6: "Bare Ground",
    7: "Snow/Ice",
    8: "Clouds",
    9: "Rangeland"
}

In [6]:
DIR_IMAGES = Path("C:\\Users\\remot\\Documents\\Jerome\\03_Independent_Research\\20260630 Mangrove and Seagrass Monitoring\\03_Data_management\\01_Raw_data\\Rasters")
fps_images = list(DIR_IMAGES.glob("*_PlanetScope8band_*"))

# Only use a specific date for Hinatuan
fps_images = [fp for fp in fps_images if "Hinatuan" not in fp.stem or "20210729" in fp.stem]

for fp in fps_images:
    print(fp)

C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\01_Raw_data\Rasters\Busuanga-Palawan_PlanetScope8band_20240421.zip
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\01_Raw_data\Rasters\Calauag-Quezon_PlanetScope8band_20250804
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\01_Raw_data\Rasters\DumaranIsland-Palawan_PlanetScope8band_20220524
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\01_Raw_data\Rasters\Hinatuan-SurigaoDelSur_PlanetScope8band_20210729
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\01_Raw_data\Rasters\Infanta-Quezon_PlanetScope8band_20210903
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Mo

In [7]:
DIR_AOIs = Path().resolve().parent / 'output'
fps_aoi = list(DIR_AOIs.glob("*_AOI_MangroveSeagrassMapping*"))

aoi_names = [fp.stem.split("-")[0] for fp in fps_images]
fps_aoi = [fp for fp in fps_aoi if fp.stem.split("_")[0] in aoi_names]

for fp in fps_aoi:
    print(fp)

C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\03_Scripts\output\Busuanga_AOI_MangroveSeagrassMapping.geojson
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\03_Scripts\output\Calauag_AOI_MangroveSeagrassMapping.geojson
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\03_Scripts\output\DumaranIsland_AOI_MangroveSeagrassMapping.geojson
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\03_Scripts\output\Hinatuan_AOI_MangroveSeagrassMapping.geojson
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\03_Scripts\output\Infanta_AOI_MangroveSeagrassMapping.geojson
C:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring

In [ ]:
client = Client(processes=False)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://192.168.1.4:8787/status,
Dashboard: http://192.168.1.4:8787/status,Workers: 1
Total threads: 20,Total memory: 15.63 GiB
Status: running,Using processes: False
Comm: inproc://192.168.1.4/5928/1,Workers: 0
Dashboard: http://192.168.1.4:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: inproc://192.168.1.4/5928/4,Total threads: 20
Dashboard: http://192.168.1.4:62277/status,Memory: 15.63 GiB
Nanny: None,


2026-11-06 19:45:16,724 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle ded25dfa086a28d7e8575330a9428b7e initialized by task ('rechunk-merge-rechunk-transfer-c49e4007b6db9f6d5e37fcddc3b9a71f', 127, 0, 127, 0) executed on worker inproc://192.168.1.4/5928/4
2026-11-06 19:45:24,906 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 08f923c45e7a8ef41922f7ab91f4f49a initialized by task ('rechunk-merge-rechunk-transfer-c49e4007b6db9f6d5e37fcddc3b9a71f', 126, 0, 126, 6) executed on worker inproc://192.168.1.4/5928/4
2026-11-06 19:45:28,394 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle ded25dfa086a28d7e8575330a9428b7e deactivated due to stimulus 'task-finished-1793965528.3898153'
2026-11-06 19:45:39,032 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle ea18d836003305ea5c1c7681960dd44e initialized by task ('rechunk-merge-rechunk-transfer-c49e4007b6db9f6d5e37fcddc3b9a71f', 123, 0, 123, 5) executed on worker inproc://192.168.1.4/5928/4
2026-11-06 19:45

1) Land cover breakdown (using ESRI 10-meter land cover)

In [9]:
api_url = 'https://planetarycomputer.microsoft.com/api/stac/v1/'
collection_id = 'io-lulc'

catalog = pystac_client.Client.open(api_url, modifier=pc.sign_inplace)

In [10]:
# configure_s3_access(aws_unsigned=True)

In [11]:
# search = catalog.search(
#     collections=[collection_id],
#     intersects=extract_aoi_geom(fps_aoi[0]),
#     datetime='2021-01-01'
# )

# items = search.item_collection()
# items

In [12]:
# ds = stac_load(
#     items,
#     resolution=10,
#     crs='utm',
#     # chunks={'x': 1024, 'y': 1024},
# )

# ds

In [13]:
# Compile all AOIs in one DataFrame
df_aoi_lulc = pd.concat([gpd.read_file(fp) for fp in fps_aoi], ignore_index=True)
df_aoi_lulc['aoi'] = [fp.stem.split("_")[0] for fp in fps_aoi]

# Convert to GeoDataFrame
gdf_aoi_lulc = gpd.GeoDataFrame(data=df_aoi_lulc, geometry=df_aoi_lulc['geometry']).drop(columns=['layer', 'path'])
gdf_aoi_lulc = gdf_aoi_lulc[['aoi', 'geometry']]

# Convert GeometryCollection to MultiPolygon
# is_geomcoll = gdf_aoi_lulc.geom_type == 'GeometryCollection'
polygons = [p for p in gdf_aoi_lulc.iloc[6].geometry.geoms if isinstance(p, Polygon)]
new_multipoly = MultiPolygon([p for p in polygons])
gdf_aoi_lulc.geometry = gdf_aoi_lulc.iloc[:6].geometry.to_list() + [new_multipoly]

gdf_aoi_lulc

    # aoi_name = fp.stem.split("_")[0]
    # aoi_geom = extract_aoi_geom

,aoi,geometry
0,Busuanga,"MULTIPOLYGON (((119.83166 11.98065, 119.83201 ..."
1,Calauag,"MULTIPOLYGON (((122.14541 14.17514, 122.14571 ..."
2,DumaranIsland,"MULTIPOLYGON (((119.88771 10.54809, 119.88827 ..."
3,Hinatuan,"MULTIPOLYGON (((126.26826 8.31473, 126.26896 8..."
4,Infanta,"MULTIPOLYGON (((121.61302 14.56564, 121.61201 ..."
5,RoxasCity,"MULTIPOLYGON (((122.67848 11.56349, 122.67709 ..."
6,ZamboangaCity,"MULTIPOLYGON (((122.0571 6.87274, 122.06434 6...."


In [ ]:
results_per_aoi = []

for _, row in gdf_aoi_lulc.iterrows():
    aoi_name = row['aoi']
    geometry = make_valid(row['geometry'], method='structure')

    print(f'Processing {aoi_name}', end='... ')
    # print(f'Processing {aoi_name}...')

    da_lulc = extract_lulc_aoi(geometry, catalog, collection_id)
    # transform = da_lulc.rio.transform()

    # ddf = dd.from_dask_array(da_lulc.data.ravel(), columns=['pixel_value'])
    # counts = ddf['pixel_value'].value_counts()

    # final_counts = counts.compute()

    # dask_arr = da_lulc.data

    # max_val = 11

    # counts = np.bincount(dask_arr.ravel(), minlength=max_val + 1)
    # frequencies = counts.compute()
    # unique_values = np.arange(len(frequencies))

    unique_targets = np.array(list(LAND_COVER_CATEGORIES.keys()))
    bins = np.append(unique_targets - 0.5, unique_targets[-1] + 0.5)
    
    pixel_counts = histogram(da_lulc, bins=[bins], dim=['x', 'y'])
    # print(pixel_counts.compute())

    # for lulc, freq in zip(unique_values, frequencies):
    #     gdf_aoi_lulc.loc[gdf_aoi_lulc['aoi'] == aoi_name, lulc] = freq

    # print('Done!')

    # results = rstat.zonal_stats(
    #     vectors=[geometry],
    #     raster=da_lulc.values,
    #     affine=transform,
    #     categorical=True,
    #     nodata=0
    # )

    # for lulc in results[0]:
    #     gdf_aoi_lulc.loc[gdf_aoi_lulc['aoi'] == aoi_name, lulc] = results[0][lulc]
    results_per_aoi.append(pixel_counts)
    print('Done!')

results_per_aoi = dask.compute(*results_per_aoi)
results_per_aoi[0]

Processing Busuanga... Done!
Processing Calauag... Done!
Processing DumaranIsland... Done!
Processing Hinatuan... Done!
Processing Infanta... Done!
Processing RoxasCity... Done!
Processing ZamboangaCity... Done!


c:\Users\remot\Documents\Jerome\03_Independent_Research\20260630 Mangrove and Seagrass Monitoring\03_Data_management\03_Scripts\.venv\Lib\site-packages\distributed\client.py:3429: UserWarning: Sending large graph of size 17.53 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Exception: RuntimeError("Error reading Window(col_off=50176, row_off=124928, width=1024, height=1024) from 'https://ai4edataeuwest.blob.core.windows.net/io-lulc/io-lulc-model-001-v01-composite-v03-supercell-v02-clip-v01/51N_20200101-20210101.tif?st=2026-11-05T11%3A00%3A56Z&se=2026-11-06T11%3A45%3A56Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-11-05T01%3A21%3A05Z&ske=2026-11-12T01%3A21%3A05Z&sks=b&skv=2025-07-05&sig=UG2249dsuy4/wFYZfmXYO0P4yno9VXr2%2BZ0uScNkoWQ%3D': RasterioIOError('Read failed. See previous exception for details.')")

In [ ]:
unique_targets

array(dict_keys([1, 2, 3, 4, 5, 6, 7, 8, 9]), dtype=object)

In [ ]:
gdf_aoi_lulc

,aoi,geometry,0,1,2,3,4,5,6,7,8,9,10,11
0,Busuanga,"MULTIPOLYGON (((119.83166 11.98065, 119.83201 ...",1.041851e+10,1.747100e+09,942079801.0,24553637.0,8175149.0,225943365.0,152317243.0,85827134.0,3277112.0,12019.0,2817532.0,0.0
1,Calauag,"MULTIPOLYGON (((122.14541 14.17514, 122.14571 ...",4.904888e+09,1.515377e+09,869788112.0,23453906.0,7964131.0,221819711.0,143529561.0,83861422.0,3112573.0,3160.0,2471198.0,0.0
2,DumaranIsland,"MULTIPOLYGON (((119.88771 10.54809, 119.88827 ...",1.041851e+10,1.747100e+09,942079801.0,24553637.0,8175149.0,225943365.0,152317243.0,85827134.0,3277112.0,12019.0,2817532.0,0.0
3,Hinatuan,"MULTIPOLYGON (((126.26826 8.31473, 126.26896 8...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Infanta,"MULTIPOLYGON (((121.61302 14.56564, 121.61201 ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,RoxasCity,"MULTIPOLYGON (((122.67848 11.56349, 122.67709 ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ZamboangaCity,"MULTIPOLYGON (((122.0571 6.87274, 122.06434 6....",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# for aoi in aoi_names:
#     da_chunked = ds_per_aoi[aoi].data
    
#     # Get unique values and counts lazily
#     unique_values, counts = da.unique(da_chunked.data, return_counts=True)

#     with dask.config.set({'optimization.fuse.active': True}):
#         with dask.config.set(num_workers=4):
#             val_counts = xr.DataArray(
#                 counts,
#                 dims=['value'],
#                 coords={'value': unique_values}
#             )

#             result = val_counts.compute()

#     print(aoi)
#     print(result)


Exception: RasterioIOError('HTTP response code: 409')